## Workload processing

In [22]:
import pandas as pd
import re
import numpy as np

def clean_and_convert(text):
    original = str(text)
    if pd.isna(text) or original.strip() == '' or original.lower() == 'nan':
        return pd.Series([np.nan, "Original Missing"])
    
    t = original.lower().strip()

    # 1. Handle ranges (e.g., "2-4") immediately
    range_pattern = r'(\d+(?:\.\d+)?)\s*[-–—~至]\s*(\d+(?:\.\d+)?)'
    def replace_range(m):
        return str((float(m.group(1)) + float(m.group(2))) / 2)
    t = re.sub(range_pattern, replace_range, t)

    # 2. Check for "Hours per Week" pattern
    # Matches: "2.5 hours a week", "3h/week", "4 hours/w"
    per_week_match = re.search(r'(\d+(?:\.\d+)?)\s*(?:hour|h|hr|小时|час).*(?:per|a|/|每周)\s*(?:week|w|周|недел)', t)
    
    # Check for "Total Weeks" pattern 
    # Matches: "4 weeks", "5 semanas"
    weeks_match = re.search(r'(\d+(?:\.\d+)?)\s*(?:week|sem|周|недел)', t)

    if per_week_match and weeks_match:
        # Scenario: "4 weeks... 3 hours a week" -> 4 * 3 = 12
        hours_per_week = float(per_week_match.group(1))
        total_weeks = float(weeks_match.group(1))
        return pd.Series([hours_per_week * total_weeks, "Success (Weekly Calculation)"])

    # 3. Handle "X hours Y minutes" (e.g., 4h 30m)
    # This regex looks for hours and minutes specifically to combine them
    h_m_match = re.search(r'(\d+(?:\.\d+)?)\s*(?:hour|h|hr|小时|час)\s*(\d+(?:\.\d+)?)\s*(?:min|m|分)', t)
    if h_m_match:
        total = float(h_m_match.group(1)) + (float(h_m_match.group(2)) / 60)
        return pd.Series([total, "Success (H+M Combination)"])

    # 4. Standard Unit Extraction (Fallbacks)
    # Total Weeks only
    if weeks_match:
        return pd.Series([float(weeks_match.group(1)) * 10, "Success (Total Weeks Only)"])
    
    # Total Hours only
    hours_only = re.search(r'(\d+(?:\.\d+)?)\s*(?:hour|h|hr|小时|час)', t)
    if hours_only:
        return pd.Series([float(hours_only.group(1)), "Success (Total Hours)"])
    
    # Minutes only
    mins_only = re.search(r'(\d+(?:\.\d+)?)\s*(?:min|m|分)', t)
    if mins_only:
        return pd.Series([float(mins_only.group(1)) / 60, "Success (Minutes Only)"])

    # 5. Last Resort: Just a number
    just_num = re.search(r'(\d+(?:\.\d+)?)', t)
    if just_num:
        return pd.Series([float(just_num.group(1)), "Success (Numeric Fallback)"])

    return pd.Series([np.nan, "Unrecognized Format"])

In [24]:
df = pd.read_csv('coursera_full_2026.csv')
# --- Execution ---
audit_results = df['Workload'].apply(clean_and_convert)
audit_results.columns = ['Converted_Hours', 'Status']

# --- Summary & Export ---
print(audit_results['Status'].value_counts())

# Filter failed rows to see what's wrong
failed_rows = audit_results[audit_results['Status'].isin(["No Numbers Found", "Unrecognized Unit"])]
failed_rows.to_csv("conversion_failures.csv", index=False)

Status
Original Missing                7421
Success (Weekly Calculation)    4202
Success (Total Hours)           3936
Success (Total Weeks Only)      1232
Success (Minutes Only)           945
Success (Numeric Fallback)       402
Unrecognized Format              190
Success (H+M Combination)        172
Name: count, dtype: int64


In [27]:
df['Workload_Hour'] = audit_results['Converted_Hours']
df[['Workload','Workload_Hour']].head(30)

,Workload,Workload_Hour
0,NaN,NaN
1,4-8 hours/week,6.000000
2,4h 30m,4.500000
3,"4 weeks of study, 2-4 hours a week",12.000000
4,1 hour 30 minutes,1.500000
5,2 heures,2.000000
6,2 hours,2.000000
7,1 hour 30 minutes,1.500000
8,"4 weeks of study, 1-2 hours/week",6.000000
9,"Around 4 hours of videos in total, plus a fina...",4.000000
